# Lesson 4: Polymorphism in Data Science

## Learning Objectives

By the end of this lesson, you will be able to:
- Define Polymorphism and explain its benefits in writing flexible code.
- Understand 'duck typing' as Python's approach to polymorphism.
- Write a single function that can operate on objects of different classes that share a common interface.
- See how polymorphism is used in data science libraries to treat different models or transformers uniformly.

## Why This Topic Matters

You've built a set of `DataLoader` classes (`CsvLoader`, `JsonLoader`). Now, you want to process a list of different data files. Without polymorphism, you might write code like this:

```python
for loader in list_of_loaders:
    if isinstance(loader, CsvLoader):
        # do csv-specific things
    elif isinstance(loader, JsonLoader):
        # do json-specific things
    # ... and so on
```

This is rigid and clumsy. Every time you add a new loader type (e.g., `ExcelLoader`), you have to go back and add another `elif` statement. This breaks the open/closed principle of software design (your code should be open for extension, but closed for modification).

**Polymorphism** (from Greek, meaning "many forms") solves this. It's the principle that allows you to use a single interface (like a function call) to operate on objects of different classes. As long as each object's class implements the necessary methods, your code will work without needing to know the object's specific type. It lets you write flexible, generic code that doesn't need to change when you add new types.

## Intuition-First Explanation: The Power Outlet

Think of an electrical outlet on the wall. It provides a standard interface (two or three prongs).

- You can plug in a `Lamp` object.
- You can plug in a `LaptopCharger` object.
- You can plug in a `VacuumCleaner` object.

The wall outlet doesn't know or care what kind of device you're plugging in. It only cares that the device has a compatible plug. It provides power, and each device uses that power in its own way (to produce light, charge a battery, or create suction).

In this analogy:
- The **wall outlet** is your **generic function** or code.
- The **standard plug shape** is the **common interface** (the methods your objects must have, e.g., a `draw_power()` method).
- The **different appliances** are your **objects of different classes** (`Lamp`, `LaptopCharger`).

Polymorphism is the ability of the single wall outlet to work with many different forms of appliances.

## Polymorphism in Python: Duck Typing

Many languages enforce polymorphism strictly through inheritance. In Python, the approach is more flexible and is often described by the phrase:

> "If it walks like a duck and it quacks like a duck, then it must be a duck."

This is called **duck typing**. It means that Python doesn't care about an object's *type* or what class it inherits from. It only cares about whether the object has the methods or attributes that are being requested. If your code calls `obj.quack()`, Python just checks if the `obj` you gave it actually has a `quack()` method. If it does, the code runs. If not, it raises an error.

Inheritance is a great way to *ensure* that different classes have the same methods, but it's not strictly required for polymorphism to work in Python.

## Practical Example: A Universal Data Processing Function

Let's reuse our `DataLoader` classes from the previous lesson. They are a perfect example of a class hierarchy that is ready for polymorphic behavior because they all inherit from `DataLoader` and they all implement a `load_data()` method.

In [ ]:
import pandas as pd

# --- Base and Child Classes from Previous Lesson ---
class DataLoader:
    def __init__(self, source):
        self.source = source
    def load_data(self):
        raise NotImplementedError

class CsvLoader(DataLoader):
    def load_data(self):
        print("--- Using CSV Loader ---")
        from io import StringIO
        return pd.read_csv(StringIO(self.source))

class JsonLoader(DataLoader):
    def load_data(self):
        print("--- Using JSON Loader ---")
        return pd.read_json(self.source, orient='records')

# A new loader to demonstrate flexibility
class ExcelLoader(DataLoader):
    def load_data(self):
        # This is a mock; in reality, you'd use pd.read_excel(self.source)
        print("--- Using Excel Loader (Mock) ---")
        return pd.DataFrame([{'col_a': 100, 'col_b': 'X'}, {'col_a': 200, 'col_b': 'Y'}])

Now, let's write a single, polymorphic function that can process data from any of these loaders. This function is the 'wall outlet'.

In [ ]:
def process_and_summarize(loader: DataLoader):
    """
    This function takes any object that 'looks like' a DataLoader,
    loads its data, and prints a summary.
    It relies on the object having a .load_data() method.
    """
    print("Starting generic data processing...")
    
    # Polymorphic call: We don't know or care what kind of loader it is.
    # We just trust that it has a .load_data() method.
    df = loader.load_data()
    
    print("\nProcessing complete. Data summary:")
    print(df.describe())
    print("====================================\n")

# --- Create instances of our different loader classes ---
csv_data = "id,value\n1,10\n2,15\n3,20"
json_data = '[{"id": 1, "value": 100}, {"id": 2, "value": 150}]'
excel_source = 'financials.xlsx' # Mock source path

loaders = [
    CsvLoader(csv_data),
    JsonLoader(json_data),
    ExcelLoader(excel_source)
]

# --- Loop and process polymorphically ---
for l in loaders:
    # We pass different types of objects to the SAME function!
    process_and_summarize(l)

Look at how clean and simple the final loop is. The `process_and_summarize` function works with any object that respects the `DataLoader` interface (i.e., has a `load_data` method). If we invent a new `ParquetLoader` tomorrow, we don't need to change `process_and_summarize` at all. We just create the new class, and it will work automatically.

## Real-World Usage for Data Scientists

Polymorphism is the silent workhorse that makes many data science libraries so pleasant to use.

### Scikit-learn Pipelines

The `Pipeline` object in scikit-learn is a masterclass in polymorphism. A pipeline is a list of steps, where each step is a tuple of `(name, transformer)`. A transformer can be a `StandardScaler`, a `SimpleImputer`, a `PCA`, or a custom transformer you wrote yourself.

When you call `pipeline.fit_transform(data)`, the pipeline iterates through its list of transformers. For each one, it calls that transformer's `.fit_transform()` method. The pipeline doesn't have a giant `if/elif` block to check the type of each transformer. It just trusts that every object in its list has a `.fit_transform()` method (duck typing!). This is possible because they all inherit from `TransformerMixin`, which guarantees they have that common interface.

Let's simulate a mini-pipeline.

In [ ]:
import numpy as np

# --- Define some transformer classes with a common interface ---
class MeanRemover:
    """A transformer that subtracts the mean."""
    def fit_transform(self, data):
        print("Applying MeanRemover...")
        mean = np.mean(data)
        return data - mean

class Scaler:
    """A transformer that divides by the standard deviation."""
    def fit_transform(self, data):
        print("Applying Scaler...")
        std = np.std(data)
        return data / std

# --- The polymorphic pipeline function ---
def run_pipeline(data, steps):
    """Runs data through a series of transformation steps."""
    temp_data = data.copy()
    for name, transformer in steps:
        print(f"\nExecuting step: '{name}'")
        # Polymorphic call to .fit_transform()
        temp_data = transformer.fit_transform(temp_data)
    return temp_data

# --- Usage ---
my_data = np.array([10, 20, 30, 40, 50])

pipeline_steps = [
    ('remove_mean', MeanRemover()),
    ('scale_data', Scaler())
]

processed_data = run_pipeline(my_data, pipeline_steps)
print(f"\nOriginal data: {my_data}")
print(f"Processed data: {processed_data}")
print(f"Mean of processed data (should be ~0): {np.mean(processed_data)}")

Our `run_pipeline` function is polymorphic. It can work with `MeanRemover`, `Scaler`, or any future object we create, as long as that object has a `fit_transform` method.

## Common Mistakes and Misconceptions

1.  **Checking for types with `isinstance()`**: A common anti-pattern for people new to polymorphism is to explicitly check the type of an object before calling a method. 
   - **Bad**: `if isinstance(obj, CsvLoader): obj.load_data()`
   - **Good**: `obj.load_data()` (Just call the method and trust it exists).
   Using `isinstance` is sometimes necessary, but if you find yourself using it to decide which method to call, you are likely fighting against polymorphism.

2.  **Thinking it only works with inheritance**: While inheritance is the most common way to establish a shared interface, Python's duck typing means any object can be used polymorphically as long as it has the right methods. The objects don't technically need to share a parent class.

3.  **Creating inconsistent method signatures**: For polymorphism to work smoothly, the methods in the shared interface should have similar parameters. If `CsvLoader.load_data()` takes no arguments, but `JsonLoader.load_data()` requires a `chunk_size` argument, you can no longer treat them identically in a simple loop. The interface has been broken.

## Recap

- **Polymorphism**: The ability to use a single interface (a function or method) to work with objects of different classes.
- **Duck Typing**: Python's philosophy for polymorphism. If an object has the required methods/attributes ('walks and quacks like a duck'), it can be used, regardless of its actual class type.
- **Benefits**: Polymorphism leads to code that is more flexible, reusable, and extensible. You can add new functionality by adding new classes, without having to modify the functions that operate on them.
- **Shared Interface**: The set of methods and properties that a group of classes have in common, which allows them to be used polymorphically.

## Suggested Next Lesson

We've now covered the main pillars of OOP: Encapsulation, Inheritance, and Polymorphism. But Python has a few more tricks up its sleeve. The language includes a variety of special 'magic' methods (also called 'dunder' methods, for 'double underscore') like `__init__`, `__str__`, and `__len__`. Understanding these allows you to make your custom objects behave just like Python's built-in types, making them even more intuitive to use.